# Clase 7 · Deep learning para lenguaje natural

## NLP avanzado: de palabras a vectores

En esta clase construyes, paso a paso, las piezas que permiten a una red neuronal procesar texto: conviertes frases en números, entrenas una tabla de embeddings y usas esa tabla dentro de un clasificador de texto mínimo. Cierras la sesión con una idea puente hacia el mecanismo de atención, que trabajas en la próxima clase.

**Framework:** PyTorch. Ejecutas este cuaderno en Google Colab sin instalar nada adicional.

### Objetivos de aprendizaje

- Construyes un vocabulario a mano y conviertes frases en secuencias de índices
- Creas una capa `torch.nn.Embedding` y compruebas que palabras con significado parecido terminan con vectores parecidos tras entrenar
- Entrenas un clasificador de texto mínimo (embedding más capa lineal) que distingue frases positivas de negativas
- Entiendes por qué el esquema codificador-decodificador es el paso previo al mecanismo de atención

## Uno: de palabras a números

Una red neuronal solo procesa números, así que el primer paso siempre es la tokenización: divides el texto en unidades discretas (tokens) y las mapeas a un vocabulario, un conjunto finito de símbolos con un índice numérico cada uno.

- Añades dos tokens especiales: `<PAD>` para rellenar frases cortas hasta una longitud común dentro de un lote, y `<UNK>` para cualquier palabra que no esté en tu vocabulario
- Cada frase se convierte en una lista de índices de longitud fija
- Esta representación por índices todavía no dice nada sobre el significado: dos palabras parecidas tienen índices tan distintos como dos palabras sin ninguna relación. Ese problema lo resuelves con el embedding, en el siguiente bloque

In [ ]:
frases_ejemplo = [
    "el gato duerme en el sofa",
    "el perro juega en el jardin",
    "la nina lee un libro",
    "el nino escribe una carta",
]

# Construyes el vocabulario: cada palabra distinta recibe un indice.
# <PAD> rellena frases cortas; <UNK> cubre palabras no vistas.
palabras = sorted(set(palabra for frase in frases_ejemplo for palabra in frase.split()))
vocabulario = ["<PAD>", "<UNK>"] + palabras
indice_de = {palabra: i for i, palabra in enumerate(vocabulario)}

print(f"Vocabulario ({len(vocabulario)} palabras): {vocabulario}")


def tokenizar(frase, longitud_max):
    """Convierte una frase en una lista de indices, rellenada con <PAD>
    hasta longitud_max."""
    indices = [indice_de.get(p, indice_de["<UNK>"]) for p in frase.split()]
    indices = indices[:longitud_max]
    indices += [indice_de["<PAD>"]] * (longitud_max - len(indices))
    return indices


longitud_max = max(len(f.split()) for f in frases_ejemplo)
for frase in frases_ejemplo:
    print(f"{frase!r:35} -> {tokenizar(frase, longitud_max)}")

## Dos: embeddings, un mapa de significados

Imagina que colocas cada palabra como un punto en un mapa, en vez de guardarla como una etiqueta aislada. En ese mapa, las palabras con significados parecidos quedan cerca ("rey" y "reina"), y las que no tienen relación quedan lejos ("rey" y "calle").

- Ese mapa es el espacio de embeddings, y las coordenadas de cada palabra son su embedding: un vector denso de D números
- En PyTorch, `torch.nn.Embedding` implementa ese mapa como una tabla de V filas (una por palabra del vocabulario) por D columnas
- La tabla se inicializa con valores al azar y se ajusta por retropropagación durante el entrenamiento: nadie decide a mano dónde va cada palabra
- Lo compruebas entrenando un embedding sobre un corpus diminuto donde "rey" y "reina", o "hombre" y "mujer", aparecen en los mismos contextos, y midiendo la similitud entre sus vectores antes y después de entrenar

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Corpus minusculo donde "rey"/"reina" y "hombre"/"mujer" aparecen en contextos parecidos
corpus = [
    "el rey gobierna el reino",
    "la reina gobierna el reino",
    "el rey es poderoso",
    "la reina es poderosa",
    "el hombre camina por la calle",
    "la mujer camina por la calle",
    "el hombre trabaja en la oficina",
    "la mujer trabaja en la oficina",
]

palabras_emb = sorted(set(p for frase in corpus for p in frase.split()))
idx_emb = {p: i for i, p in enumerate(palabras_emb)}
V = len(palabras_emb)

# Generas pares (palabra, palabra_de_contexto) con una ventana de una posicion,
# igual que la idea de skip-gram vista en la teoria.
VENTANA = 1
pares = []
for frase in corpus:
    tokens = frase.split()
    for i, palabra in enumerate(tokens):
        for j in range(max(0, i - VENTANA), min(len(tokens), i + VENTANA + 1)):
            if i != j:
                pares.append((idx_emb[palabra], idx_emb[tokens[j]]))

entradas = torch.tensor([p[0] for p in pares], dtype=torch.long)
contextos = torch.tensor([p[1] for p in pares], dtype=torch.long)

DIM = 6
embedding = nn.Embedding(V, DIM)
salida = nn.Linear(DIM, V)  # predice la palabra de contexto a partir del embedding
optimizador = torch.optim.Adam(list(embedding.parameters()) + list(salida.parameters()), lr=0.05)


def similitud(palabra_a, palabra_b):
    va = embedding.weight[idx_emb[palabra_a]]
    vb = embedding.weight[idx_emb[palabra_b]]
    return F.cosine_similarity(va.unsqueeze(0), vb.unsqueeze(0)).item()


print("Antes de entrenar:")
print(f"  similitud(rey, reina)     = {similitud('rey', 'reina'):.3f}")
print(f"  similitud(hombre, mujer)  = {similitud('hombre', 'mujer'):.3f}")
print(f"  similitud(rey, calle)     = {similitud('rey', 'calle'):.3f}")

for epoca in range(300):
    optimizador.zero_grad()
    vectores = embedding(entradas)
    logits = salida(vectores)
    perdida = F.cross_entropy(logits, contextos)
    perdida.backward()
    optimizador.step()

print("\nDespues de entrenar:")
print(f"  similitud(rey, reina)     = {similitud('rey', 'reina'):.3f}")
print(f"  similitud(hombre, mujer)  = {similitud('hombre', 'mujer'):.3f}")
print(f"  similitud(rey, calle)     = {similitud('rey', 'calle'):.3f}")

## Tres: un clasificador de texto mínimo

Con la tokenización y el embedding ya listos, construyes un clasificador de sentimiento: recibe una frase y predice si expresa una opinión positiva o negativa.

- La arquitectura es deliberadamente simple: `nn.Embedding` para convertir cada palabra en un vector, un promedio de esos vectores para resumir la frase completa, y una capa `nn.Linear` que produce un único logit
- El promedio ignora el orden de las palabras. Es una simplificación que te permite entrenar en pocas líneas, pero pierde información: lo comprobarás en el cuaderno de práctica
- Entrenas minimizando `BCEWithLogitsLoss`, que combina una sigmoide con la entropía cruzada binaria de forma numéricamente estable
- Observas la curva de pérdida para comprobar que el modelo converge

In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt

# Frases de entrenamiento con su etiqueta: uno es positivo, cero es negativo
frases = [
    "me encanta este producto",
    "excelente servicio y rapido",
    "una experiencia maravillosa",
    "lo recomiendo totalmente",
    "el equipo fue muy amable",
    "pesimo servicio y muy lento",
    "no lo recomiendo para nada",
    "una experiencia horrible",
    "el equipo fue muy grosero",
    "producto de mala calidad",
]
etiquetas = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

palabras_clf = sorted(set(p for frase in frases for p in frase.split()))
vocab_clf = ["<PAD>", "<UNK>"] + palabras_clf
indice_clf = {p: i for i, p in enumerate(vocab_clf)}
TAMANO_VOCAB = len(vocab_clf)


def tokenizar_clf(frase, longitud_max):
    indices = [indice_clf.get(p, indice_clf["<UNK>"]) for p in frase.split()]
    indices = indices[:longitud_max]
    indices += [indice_clf["<PAD>"]] * (longitud_max - len(indices))
    return indices


LONGITUD_MAX = max(len(f.split()) for f in frases)
X = torch.tensor([tokenizar_clf(f, LONGITUD_MAX) for f in frases], dtype=torch.long)
y = torch.tensor(etiquetas, dtype=torch.float32)


class ClasificadorSentimiento(nn.Module):
    def __init__(self, tamano_vocab, dim_embedding):
        super().__init__()
        # padding_idx=0 evita que <PAD> reciba gradiente: no aporta significado
        self.embedding = nn.Embedding(tamano_vocab, dim_embedding, padding_idx=0)
        self.clasificador = nn.Linear(dim_embedding, 1)

    def forward(self, x):
        vectores = self.embedding(x)          # (lote, longitud, dim_embedding)
        promedio = vectores.mean(dim=1)        # promedio de la frase completa
        logit = self.clasificador(promedio)    # (lote, 1)
        return logit.squeeze(1)


DIM_EMBEDDING = 8
modelo = ClasificadorSentimiento(TAMANO_VOCAB, DIM_EMBEDDING)

perdida_fn = nn.BCEWithLogitsLoss()  # combina sigmoide + entropia cruzada, mas estable
optimizador = optim.Adam(modelo.parameters(), lr=0.05)

historial_perdida = []
for epoca in range(200):
    optimizador.zero_grad()
    logits = modelo(X)
    perdida = perdida_fn(logits, y)
    perdida.backward()
    optimizador.step()
    historial_perdida.append(perdida.item())
    if (epoca + 1) % 50 == 0:
        print(f"Epoca {epoca + 1}: perdida = {perdida.item():.4f}")

plt.plot(historial_perdida)
plt.xlabel("Epoca")
plt.ylabel("Perdida (BCE)")
plt.title("Curva de perdida del clasificador de sentimiento")
plt.show()

In [ ]:
def predecir(frase):
    modelo.eval()
    with torch.no_grad():
        entrada = torch.tensor([tokenizar_clf(frase, LONGITUD_MAX)], dtype=torch.long)
        probabilidad = torch.sigmoid(modelo(entrada)).item()
    etiqueta = "positivo" if probabilidad > 0.5 else "negativo"
    print(f"'{frase}' -> {etiqueta} (probabilidad = {probabilidad:.3f})")


predecir("me encanta el equipo")
predecir("una experiencia pesima")
predecir("servicio amable y rapido")

## Cuatro: hacia los modelos secuencia a secuencia

El clasificador que acabas de entrenar produce una única etiqueta a partir de una frase completa. Pero, ¿qué haces cuando la salida también es una secuencia, y de longitud distinta a la de entrada, como en la traducción automática?

- Un modelo de secuencia a secuencia combina dos redes recurrentes: un codificador que lee la frase de origen palabra a palabra y resume todo en un único vector de contexto, y un decodificador que genera la frase de destino a partir de ese vector
- El problema es el cuello de botella: toda la frase de origen, tenga cinco palabras o cincuenta, se comprime en un vector de tamaño fijo. Cuanto más larga es la frase, más información relevante se pierde
- A esto se suma la dificultad ya conocida de las redes recurrentes para relacionar posiciones muy alejadas entre sí en la secuencia
- Esta doble limitación es exactamente lo que resuelve el mecanismo de atención: en la próxima clase dejas de forzar toda la frase en un único vector y permites que el decodificador "mire hacia atrás" y pondere cada palabra de la entrada según su relevancia en cada paso

## Resumen de puntos clave

- La tokenización convierte texto en índices numéricos a partir de un vocabulario finito, con tokens especiales para relleno (`<PAD>`) y palabras desconocidas (`<UNK>`)
- Un embedding sustituye el índice disperso por un vector denso de D números, aprendido por retropropagación; palabras con significados parecidos terminan cerca en ese espacio
- Un clasificador de texto mínimo combina un embedding, un resumen de la frase (aquí, un promedio) y una capa lineal, entrenado con entropía cruzada binaria
- Promediar los embeddings es simple pero ignora el orden de las palabras; una capa recurrente conserva esa información
- Los modelos de secuencia a secuencia (codificador-decodificador) resuelven tareas donde entrada y salida tienen longitud distinta, pero sufren un cuello de botella: comprimir toda la frase en un único vector de contexto

**Abre ahora el cuaderno de práctica, `../notebooks_practica/clase07_practica_nlp_avanzado.ipynb`, y pon en práctica lo que acabas de ver.**